In [3]:
# A. Menyiapkan Streaming Source
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import window, sum as spark_sum, count, col, expr
import os
import shutil
import time
import pandas as pd
from datetime import datetime

# Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("Tugas10-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

# Menyiapkan folder simulasi streaming tugas
folder_tugas = "tugas10_stream"
shutil.rmtree(folder_tugas, ignore_errors=True)
os.makedirs(folder_tugas, exist_ok=True)

# Skema eksplisit data pesanan
skema_pesanan = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True)
])

# Mendefinisikan readStream
df_stream_tugas = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_tugas)

print("Streaming source tugas berhasil disiapkan.")

Streaming source tugas berhasil disiapkan.


In [4]:
# B. Agregasi per Cabang
# Menambahkan kolom turunan total_harga
df_transaksi = df_stream_tugas.withColumn("total_harga", col("qty") * col("harga_satuan"))

# Agregasi per cabang
agregasi_cabang = df_transaksi.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("pendapatan_total"),
    count("pesanan_id").alias("jumlah_pesanan")
)

# Menjalankan query dengan memory sink
query_cabang = agregasi_cabang.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

print("Query agregasi cabang berjalan.")

Query agregasi cabang berjalan.


In [5]:
# C. Agregasi Berbasis Window Waktu
# Membuat query kedua dengan window waktu 2 menit
agregasi_waktu = df_transaksi.groupBy(
    window(col("waktu_pesan"), "2 minutes")
).agg(
    spark_sum("total_harga").alias("pendapatan_total"),
    count("pesanan_id").alias("jumlah_pesanan")
)

query_waktu = agregasi_waktu.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()

print("Query agregasi window waktu berjalan.")

Query agregasi window waktu berjalan.


In [6]:
# D. Menguji dengan Data Simulasi
waktu_sekarang = datetime.now()

# Batch 1
pd.DataFrame({
    "pesanan_id": ["P01", "P02"],
    "nama_cabang": ["Cabang Malioboro", "Cabang Pandanaran"],
    "nama_menu": ["Kopi Tubruk", "Es Kopi Susu"],
    "qty": [2, 3],
    "harga_satuan": [15000, 18000],
    "waktu_pesan": [waktu_sekarang.strftime("%Y-%m-%d %H:%M:%S")] * 2
}).to_csv(f"{folder_tugas}/batch_1.csv", index=False)

print("Mengirim Batch 1...")
time.sleep(6)
spark.sql("SELECT * FROM agregasi_cabang").show()

# Batch 2
pd.DataFrame({
    "pesanan_id": ["P03", "P04"],
    "nama_cabang": ["Cabang Malioboro", "Cabang Simpang Lima"],
    "nama_menu": ["Americano", "Latte"],
    "qty": [1, 4],
    "harga_satuan": [20000, 25000],
    "waktu_pesan": [waktu_sekarang.strftime("%Y-%m-%d %H:%M:%S")] * 2
}).to_csv(f"{folder_tugas}/batch_2.csv", index=False)

print("Mengirim Batch 2...")
time.sleep(6)
spark.sql("SELECT * FROM agregasi_cabang").show()

# Batch 3
pd.DataFrame({
    "pesanan_id": ["P05", "P06"],
    "nama_cabang": ["Cabang Pandanaran", "Cabang Malioboro"],
    "nama_menu": ["Cappuccino", "Espresso"],
    "qty": [2, 2],
    "harga_satuan": [22000, 15000],
    "waktu_pesan": [waktu_sekarang.strftime("%Y-%m-%d %H:%M:%S")] * 2
}).to_csv(f"{folder_tugas}/batch_3.csv", index=False)

print("Mengirim Batch 3...")
time.sleep(6)
spark.sql("SELECT * FROM agregasi_cabang").show()

Mengirim Batch 1...


[Stage 1:>                  (0 + 4) / 4][Stage 3:>                  (0 + 0) / 4]

+-----------+----------------+--------------+
|nama_cabang|pendapatan_total|jumlah_pesanan|
+-----------+----------------+--------------+
+-----------+----------------+--------------+

Mengirim Batch 2...


+-------------------+----------------+--------------+
|        nama_cabang|pendapatan_total|jumlah_pesanan|
+-------------------+----------------+--------------+
|Cabang Simpang Lima|          100000|             1|
|  Cabang Pandanaran|           54000|             1|
|   Cabang Malioboro|           50000|             2|
+-------------------+----------------+--------------+

Mengirim Batch 3...


+-------------------+----------------+--------------+
|        nama_cabang|pendapatan_total|jumlah_pesanan|
+-------------------+----------------+--------------+
|Cabang Simpang Lima|          100000|             1|
|  Cabang Pandanaran|           98000|             2|
|   Cabang Malioboro|           80000|             3|
+-------------------+----------------+--------------+



In [7]:
# E. Sink Berbasis Berkas
# Membersihkan folder output & checkpoint jika sebelumnya ada
shutil.rmtree("tugas10_output", ignore_errors=True)
shutil.rmtree("tugas10_checkpoint", ignore_errors=True)

query3 = df_stream_tugas.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()

print("File sink query (Parquet) dimulai.")

File sink query (Parquet) dimulai.


In [8]:
# Kirim satu batch data tambahan untuk memicu file sink Parquet, lalu verifikasi hasilnya
# Kirim batch tambahan untuk file sink
pd.DataFrame({
    "pesanan_id": ["P07"],
    "nama_cabang": ["Cabang Malioboro"],
    "nama_menu": ["V60"],
    "qty": [1],
    "harga_satuan": [20000],
    "waktu_pesan": [waktu_sekarang.strftime("%Y-%m-%d %H:%M:%S")]
}).to_csv(f"{folder_tugas}/batch_4.csv", index=False)

time.sleep(6)

# Memeriksa isi folder output
print("Isi folder tugas10_output:")
!ls tugas10_output

# Membaca kembali berkas Parquet sebagai DataFrame batch biasa
df_hasil_parquet = spark.read.parquet("tugas10_output")
print("Data terbaca kembali dari file Parquet:")
df_hasil_parquet.show()

Isi folder tugas10_output:
part-00000-8ca1580f-6971-4d7b-a40f-e87e73ce8057-c000.snappy.parquet
part-00000-e28890ca-5069-4fa7-b737-7a86c0034799-c000.snappy.parquet
part-00001-1ed1fe67-c881-43a1-8341-0b501f41c7c4-c000.snappy.parquet
part-00002-4ce259e9-68d0-4c38-915c-327a44840cec-c000.snappy.parquet
_spark_metadata
Data terbaca kembali dari file Parquet:
+----------+-------------------+------------+---+------------+-------------------+
|pesanan_id|        nama_cabang|   nama_menu|qty|harga_satuan|        waktu_pesan|
+----------+-------------------+------------+---+------------+-------------------+
|       P05|  Cabang Pandanaran|  Cappuccino|  2|       22000|2026-11-05 03:35:57|
|       P06|   Cabang Malioboro|    Espresso|  2|       15000|2026-11-05 03:35:57|
|       P01|   Cabang Malioboro| Kopi Tubruk|  2|       15000|2026-11-05 03:35:57|
|       P02|  Cabang Pandanaran|Es Kopi Susu|  3|       18000|2026-11-05 03:35:57|
|       P03|   Cabang Malioboro|   Americano|  1|       20000|20

In [9]:
# Clean Up
query_cabang.stop()
query_waktu.stop()
query3.stop()
spark.stop()
print("Seluruh streaming query tugas berhasil dihentikan dan sesi ditutup.")

Seluruh streaming query tugas berhasil dihentikan dan sesi ditutup.
